# Section 6 tables

Produces Tables 1–5 of the paper for the eight TPC-DS workloads. Each table is displayed, then written to
`TABLES_DIR` as `.csv` and as a booktabs `.tex` (requires `booktabs`, `graphicx` and `makecell`).

| Table | Content | Source |
|---|---|---|
| 1 | Query text and operator structure | static analysis of `workloads/<w>/q*.sql` |
| 2 | Workload runtime distribution | `runtimes.csv` of each execution run |
| 3 | Generation overhead | `workloads/<w>/generation_report.json` |
| 4 | Structural diversity of compiled plans | `workload_metrics.csv` (`analyse_diversity.ipynb`) + GIN Vendi (`check_cosine_vendi.ipynb`) |
| 5 | Schema diversity | `workload_metrics.csv` (`analyse_diversity.ipynb`) |

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.py").exists())
for _p in (str(ROOT), str(Path.cwd())):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from config import TABLES_DIR
from diversql.shared.sql_features import query_features, _HAVE_SQLGLOT
from workloads import (
    REF, OURS, ROWS, GENERATORS, PRETTY, PLAN_METRICS_KEY, GIN_VENDI_CSV, GIN_VENDI_COL,
    load_sql, load_generation_report, load_runtimes, load_workload_metrics,
)

TABLES_DIR.mkdir(parents=True, exist_ok=True)

# Without sqlglot, query_features falls back to a regex tokeniser and Table 1's counts change.
assert _HAVE_SQLGLOT, "install sqlglot (requirements.txt) to reproduce Table 1"
_INV = {v: k for k, v in PRETTY.items()}


def show(tbl, fmts):
    # Notebook display: reference row italic/grey, DiverSQL rows bold/orange.
    def hl(row):
        key = _INV.get(row.name, row.name)
        if key == REF:
            return ["background-color:#f3f4f6;font-style:italic"] * len(row)
        if key in OURS:
            return ["background-color:#fff1e6;font-weight:600"] * len(row)
        return [""] * len(row)
    return tbl.rename(index=PRETTY).style.apply(hl, axis=1).format(fmts, na_rep="--")

## LaTeX export

Rows follow `tbl.index` (reference, baselines, DiverSQL). The reference row is italic and separated by a rule,
DiverSQL rows are bold. With `highlight=True` the best value per column among the generators is bold and
underlined; ties on the formatted value are all marked, and a column where every generator ties is not marked.

In [ ]:
_LATEX_SPECIAL = {"%": r"\%", "&": r"\&", "_": r"\_", "#": r"\#"}


def _tex_escape(s):
    for ch, esc in _LATEX_SPECIAL.items():
        s = s.replace(ch, esc)
    return s


def _tex_header(s):
    # '\n' in a header label becomes a line break inside \makecell.
    lines = s.split("\n")
    if len(lines) == 1:
        return _tex_escape(lines[0])
    return r"\makecell{" + r" \\ ".join(_tex_escape(line) for line in lines) + "}"


def to_latex(tbl, fmts, headers, caption, label, *, highlight=False, higher_is_better=True,
             placement="t", vspace=None):
    cols = list(fmts)
    hib = higher_is_better if isinstance(higher_is_better, dict) else {c: higher_is_better for c in cols}

    best_disp = {}
    if highlight:
        competitors = [w for w in tbl.index if w != REF]
        for c in cols:
            vals = tbl.loc[competitors, c].astype(float).dropna()
            if len(vals) >= 2:
                disp = fmts[c].format(vals.max() if hib.get(c, True) else vals.min())
                if any(fmts[c].format(v) != disp for v in vals):
                    best_disp[c] = disp

    def fmt_cell(w, c):
        v = tbl.loc[w, c]
        if pd.isna(v):
            return "--"
        s = fmts[c].format(float(v))
        is_best = w != REF and best_disp.get(c) == s
        s = _tex_escape(s)
        return r"\underline{\textbf{" + s + "}}" if is_best else s

    body = [
        r"\begin{tabular}{l" + "c" * len(cols) + "}",
        r"\toprule",
        "Workload & " + " & ".join(_tex_header(headers.get(c, c)) for c in cols) + r" \\",
        r"\midrule",
    ]
    for w in tbl.index:
        name = PRETTY.get(w, w)
        if w == REF:
            name = r"\textit{" + name + "}"
        elif w in OURS:
            name = r"\textbf{" + name + "}"
        body.append(f"{name} & " + " & ".join(fmt_cell(w, c) for c in cols) + r" \\")
        if w == REF:
            body.append(r"\midrule")
    body += [r"\bottomrule", r"\end{tabular}"]

    return "\n".join([
        r"\begin{table}[" + placement + "]",
        r"\centering",
        r"\caption{" + _tex_escape(caption) + "}",
        *([r"\vspace{" + vspace + "}"] if vspace else []),
        r"\label{" + label + "}",
        r"\resizebox{\linewidth}{!}{%",
        *body,
        r"}",
        r"\end{table}",
    ])


def export(tbl, name, latex):
    tbl.rename(index=PRETTY).to_csv(TABLES_DIR / f"{name}.csv")
    (TABLES_DIR / f"{name}.tex").write_text(latex)
    print(latex)

## Table 1 · Query text and operator structure

Per-query means from static analysis of the SQL text (`query_features`). *Table Scan* is tables referenced,
*Select* is predicates, and *Nesting* is subqueries plus CTEs.

In [ ]:
features = {w: [query_features(sql) for sql in load_sql(w).values()] for w in ROWS}


def _mean(w, key):
    return float(np.mean([f.get(key, 0) or 0 for f in features[w]]))


table1 = pd.DataFrame({
    "Length (chars)": [_mean(w, "query_length_chars") for w in ROWS],
    "Table Scan": [_mean(w, "num_tables") for w in ROWS],
    "Join": [_mean(w, "num_joins") for w in ROWS],
    "Sort": [_mean(w, "num_order_by") for w in ROWS],
    "GroupBy": [_mean(w, "num_group_by") for w in ROWS],
    "Select": [_mean(w, "num_predicates") for w in ROWS],
    "Window": [_mean(w, "num_window_functions") for w in ROWS],
    "Nesting": [float(np.mean([f["num_subqueries"] + f["num_ctes"] for f in features[w]])) for w in ROWS],
}, index=ROWS)

FMTS1 = {"Length (chars)": "{:.0f}", "Table Scan": "{:.2f}", "Join": "{:.2f}", "Sort": "{:.2f}",
         "GroupBy": "{:.2f}", "Select": "{:.2f}", "Window": "{:.3f}", "Nesting": "{:.2f}"}
show(table1, FMTS1)

In [ ]:
export(table1, "table1_query_composition", to_latex(
    table1, FMTS1,
    headers={"Length (chars)": "Length\n(chars)", "Table Scan": "Table\nScan"},
    caption="Query text and operator structure on TPC-DS.",
    label="tab:operator-structure",
))

## Table 2 · Workload runtime distribution

Per-query runtimes from each workload's single sequential execution; failed and timed-out queries are excluded.
The TPC-DS run covers 1,208 queries and is clipped to `q1`–`q1000`.

In [ ]:
runtimes = {w: load_runtimes(w, max_qid=1000 if w == REF else None) for w in ROWS}

table2 = pd.DataFrame({
    "Min runtime (s)": [runtimes[w].min() for w in ROWS],
    "Mean runtime (s)": [runtimes[w].mean() for w in ROWS],
    "Median runtime (s)": [np.median(runtimes[w]) for w in ROWS],
    "P90 runtime (s)": [np.percentile(runtimes[w], 90) for w in ROWS],
    "Max runtime (s)": [runtimes[w].max() for w in ROWS],
}, index=ROWS)

FMTS2 = {"Min runtime (s)": "{:.3f}", "Mean runtime (s)": "{:.2f}", "Median runtime (s)": "{:.2f}",
         "P90 runtime (s)": "{:.2f}", "Max runtime (s)": "{:.1f}"}
show(table2, FMTS2)

In [ ]:
export(table2, "table2_runtime_distribution", to_latex(
    table2, FMTS2,
    headers={c: c.replace(" runtime", "\nruntime") for c in FMTS2},
    caption="Workload runtime distribution on TPC-DS.",
    label="tab:workload-taxonomy",
))

## Table 3 · Generation overhead

Totals over every LLM call a generator issued, including discarded candidates, divided by the 1,000 delivered
queries. The TPC-DS reference is not generated by an LLM, so it has no row.

In [ ]:
def _overhead(w):
    r = load_generation_report(w)
    n, usage, minutes = r["num_queries"], r["token_usage"], r["duration_s"] / 60
    return {
        "Tokens / query": usage["total_tokens"] / n,
        "Input tokens / query": usage["input_tokens"] / n,
        "Output tokens / query": usage["output_tokens"] / n,
        "LLM calls / query": usage["calls"] / n,
        "Gen. time (min)": minutes,
        "Queries / min": n / minutes,
    }


table3 = pd.DataFrame([_overhead(w) for w in GENERATORS], index=GENERATORS)

FMTS3 = {"Tokens / query": "{:,.0f}", "Input tokens / query": "{:,.0f}", "Output tokens / query": "{:,.0f}",
         "LLM calls / query": "{:.2f}", "Gen. time (min)": "{:.1f}", "Queries / min": "{:.2f}"}
show(table3, FMTS3)

In [ ]:
export(table3, "table3_generation_overhead", to_latex(
    table3, FMTS3,
    headers={"Tokens / query": "Tokens /\nquery", "Input tokens / query": "Input tokens\n/ query",
             "Output tokens / query": "Output tokens\n/ query", "LLM calls / query": "LLM calls\n/ query",
             "Gen. time (min)": "Gen. time\n(min)", "Queries / min": "Queries\n/ min"},
    caption="Generation overhead.",
    label="tab:quality-overhead",
))

## Table 4 · Structural diversity of compiled plans

*Operator coverage* divides the operator count by the 57 non-DML Trino operator types in
`diversql/resources/operators_ground_truth.csv`. **Vendi (κ)** uses the weighted-Jaccard plan-graph kernel; **Vendi (cosine)**
uses GIN plan embeddings and is read from `check_cosine_vendi.ipynb`'s output.

In [ ]:
metrics = load_workload_metrics()

_ops = pd.read_csv(ROOT / "diversql" / "resources" / "operators_ground_truth.csv")
N_OPERATORS = int((_ops["category"].astype(str).str.lower() != "dml").sum())
assert N_OPERATORS == 57, N_OPERATORS

_gin = pd.read_csv(GIN_VENDI_CSV, index_col=0)

table4 = pd.DataFrame({
    "Operator count": metrics["n_operator_types_observed"],
    "Operator coverage": metrics["n_operator_types_observed"] / N_OPERATORS,
    "Unique plan-graph ratio": metrics["plan_graph_uniqueness_ratio"],
    "Mean NN distance": metrics["mean_nearest_neighbour_distance"],
    "Vendi (kappa)": metrics["plan_graph_vendi_score"],
    "Vendi (cosine)": [_gin.loc[PLAN_METRICS_KEY.get(w, w), GIN_VENDI_COL] for w in ROWS],
}, index=ROWS).astype(float)

FMTS4 = {"Operator count": "{:.0f}", "Operator coverage": "{:.2f}", "Unique plan-graph ratio": "{:.3f}",
         "Mean NN distance": "{:.3f}", "Vendi (kappa)": "{:.1f}", "Vendi (cosine)": "{:.1f}"}
show(table4, FMTS4)

In [ ]:
export(table4, "table4_structural_diversity", to_latex(
    table4, FMTS4,
    headers={"Operator count": "Operator\ncount", "Operator coverage": "Operator\ncoverage",
             "Unique plan-graph ratio": "Unique plan-\ngraph ratio", "Mean NN distance": "Mean NN\ndistance",
             "Vendi (kappa)": "Vendi\n($\\kappa$)", "Vendi (cosine)": "Vendi\n(cosine)"},
    caption="Structural diversity of compiled plans.",
    label="tab:structural-diversity",
    highlight=True,
    placement="h!",
    vspace="-1em",
))

## Table 5 · Schema diversity

Fraction of the schema's tables, columns and foreign-key joins referenced anywhere in the workload, and the
fraction of queries with a distinct table set.

In [ ]:
table5 = pd.DataFrame({
    "Table coverage": metrics["table_coverage"],
    "Column coverage": metrics["column_coverage"],
    "Join-edge coverage": metrics["join_edge_coverage"],
    "Unique table-set ratio": metrics["unique_table_set_ratio"],
}, index=ROWS).astype(float)

FMTS5 = {c: "{:.3f}" for c in table5.columns}
show(table5, FMTS5)

In [ ]:
export(table5, "table5_schema_diversity", to_latex(
    table5, FMTS5,
    headers={"Table coverage": "Table\ncoverage", "Column coverage": "Column\ncoverage",
             "Join-edge coverage": "Join-edge\ncoverage", "Unique table-set ratio": "Unique table\n-set ratio"},
    caption="Schema diversity: coverage and table-combination breadth on TPC-DS.",
    label="tab:schema-diversity",
    highlight=True,
))